### RAG Application using TypeSense

In [2]:
import typesense

In [8]:
client=typesense.Client({
  'nodes': [{
    'host': 'fzm31sc0lteb4dwyp-1.a1.typesense.net',  # For Typesense Cloud use xxx.a1.typesense.net
    'port': '443',       # For Typesense Cloud use 443
    'protocol': 'https'    # For Typesense Cloud use https
  }],
  'api_key':'x8bL2vguZ2uO2hDCOkWvnKCX27V0IhbQ',
  'connection_timeout_seconds': 2
})

books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}
existing_collections = [c['name'] for c in client.collections.retrieve()]
if books_schema['name'] in existing_collections:
  print(f"Collection `{books_schema['name']}` already exists. Skipping create.")
else:
  print(client.collections.create(books_schema))

{'created_at': 1776360227, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': T

In [9]:
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [10]:
search_parameters={
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [11]:
search_parameters = {
  'q'         : 'harry potter',
  'query_by'  : 'title',
  'filter_by' : 'publication_year:<1998',
  'sort_by'   : 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}}],
 'out_of': 9979,
 'page': 1,
 'request_params': {'collection_name

In [12]:
search_parameters = {
  'q'         : 'wimpy kid',
  'query_by'  : 'title',
  'facet_by'  : 'authors',
  'sort_by'   : 'average_rating:desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [{'counts': [{'count': 1,
     'highlighted': 'Jeff Kinney',
     'value': 'Jeff Kinney'}],
   'field_name': 'authors',
   'sampled': False,
   'stats': {'total_values': 1}}],
 'found': 1,
 'hits': [{'document': {'authors': ['Jeff Kinney'],
    'average_rating': 4.16,
    'id': '5353',
    'image_url': 'https://images.gr-assets.com/books/1328811516m/7528717.jpg',
    'publication_year': 2010,
    'ratings_count': 15956,
    'title': 'The Wimpy Kid Movie Diary: How Greg Heffley Went Hollywood'},
   'highlight': {'title': {'matched_tokens': ['Wimpy', 'Kid'],
     'snippet': 'The <mark>Wimpy</mark> <mark>Kid</mark> Movie Diary: How Greg Heffley Went Hollywood'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Wimpy', 'Kid'],
     'snippet': 'The <mark>Wimpy</mark> <mark>Kid</mark> Movie Diary: How Greg Heffley Went Hollywood'}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
   

In [14]:
### Langchain + Typsense + Groq LLM + RAG Application
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [16]:
import os
os.environ["GROQ_API_KEY"] = ""

In [17]:
loader = TextLoader("test.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

C:\Users\tiwar\AppData\Local\Temp\ipykernel_24124\13807947.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
C:\Users\tiwar\AppData\Local\Temp\ipykernel_24124\13807947.py:6: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
c:\Users\tiwar\RAG\.venv-1\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine 

In [20]:
docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        'host': 'fzm31sc0lteb4dwyp-1.a1.typesense.net',  # For Typesense Cloud use xxx.a1.typesense.net
        'port': '443',       # For Typesense Cloud use 443
        'protocol': 'https',    # For Typesense Cloud use https
        "typesense_api_key": "x8bL2vguZ2uO2hDCOkWvnKCX27V0IhbQ",
        "typesense_collection_name": "lang-chain"
    },
    
)

In [21]:
query = "What is artificial intelligence"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Artificial Intelligence: A Long-Form Overview

Artificial Intelligence (AI) is the field of computing focused on building systems that can perform tasks that usually require human intelligence. These tasks include understanding language, recognizing patterns, learning from data, making predictions, planning actions, and generating new content. AI is not a single technology. It is a broad discipline that combines computer science, mathematics, statistics, optimization, neuroscience-inspired ideas, and large-scale engineering.

In simple terms, AI tries to answer one practical question: how can machines perceive, reason, and act effectively in complex environments? Over time, researchers have built many different approaches to answer that question. Some methods rely on hard-coded rules. Others rely on probability and statistical models. Modern AI is heavily shaped by machine learning, where models learn useful behavior directly from data rather than relying only on manual instructions.


In [22]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000001433F6E3C90>, search_kwargs={})

In [23]:
query = "Artificial intelligence indepth explanation"
retriever.invoke(query)[0]

Document(metadata={'source': 'test.txt'}, page_content='Artificial Intelligence: A Long-Form Overview\n\nArtificial Intelligence (AI) is the field of computing focused on building systems that can perform tasks that usually require human intelligence. These tasks include understanding language, recognizing patterns, learning from data, making predictions, planning actions, and generating new content. AI is not a single technology. It is a broad discipline that combines computer science, mathematics, statistics, optimization, neuroscience-inspired ideas, and large-scale engineering.\n\nIn simple terms, AI tries to answer one practical question: how can machines perceive, reason, and act effectively in complex environments? Over time, researchers have built many different approaches to answer that question. Some methods rely on hard-coded rules. Others rely on probability and statistical models. Modern AI is heavily shaped by machine learning, where models learn useful behavior directly 